# Capstone Two — Data Wrangling
**Project:** Predicting Residential Solar Adoption Across U.S. Counties
**Author:** Colin Segura — Springboard Data Science Career Track

This notebook covers Step 2 of the capstone (data wrangling) and follows the four
substeps from the Data Science Method: **Data Collection, Data Organization, Data
Definition, and Data Cleaning**.

The end product is a single clean county-level dataframe combining:

- **LBNL Tracking the Sun** — installation-level residential PV data, aggregated up to county-level adoption counts (the modeling target)
- **U.S. Census ACS** — county-level housing, income, and population (denominator and key features)
- **EIA Open Data** — state-level residential retail electricity prices (key feature)

NREL irradiance and DSIRE policy indicators are deferred to the feature-engineering step.

> **Note on data files.** This notebook reads from CSVs in the same folder
> as the notebook itself (flat layout — no subfolders required). In the live
> project these are downloaded from the source URLs listed in the project
> proposal. The synthetic samples used here are schema-faithful to the real
> sources, so the wrangling code below runs unchanged against the real downloads.


## 1. Data Collection

Goal: load each raw dataset into a pandas DataFrame. The three sources arrive
in completely different shapes — installation-level, county-level wide, and
state-year long — so each needs its own load step before they can be joined.


In [1]:
import os
import glob
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

RAW_DIR = Path(".")
PROCESSED_DIR = Path(".")

# Quick look at the working directory
for f in sorted(glob.glob(str(RAW_DIR / "*.csv"))):
    size_kb = os.path.getsize(f) / 1024
    print(f"{Path(f).name:42s}  {size_kb:8.1f} KB")


acs_county_housing_income.csv                    3.8 KB
eia_state_electricity_prices.csv                11.5 KB
tracking_the_sun_sample.csv                    969.2 KB


### 1.1 Load Tracking the Sun (target source)

A note on FIPS codes: county FIPS are 5-digit strings, but several states have
codes that begin with `0` (California `06xxx`, Colorado `08xxx`, Connecticut
`09xxx`, etc). If pandas reads them as integers the leading zero gets dropped
and joins to ACS silently fail. Force the column to string on read.


In [2]:
tts = pd.read_csv(
    RAW_DIR / "tracking_the_sun_sample.csv",
    dtype={"county_fips": "string", "zip_code": "string"},
)
print(f"Tracking the Sun: {tts.shape[0]:,} rows, {tts.shape[1]} columns")
tts.head()


Tracking the Sun: 12,724 rows, 10 columns


In [3]:
print(tts.head().to_string())

  data_provider_1 system_id_1 installation_date  system_size_DC  total_installed_price customer_segment state zip_code              county county_fips
0     PROVIDER_CA     S184759        2015-09-06            4.24                19297.0              RES    CA    80340  Los Angeles County       06037
1     PROVIDER_CA     S730469        2020-05-23            8.67                29099.0              RES    CA    41907  Los Angeles County       06037
2     PROVIDER_CA     S903808        2015-12-13            6.43                28029.0              RES    CA    42998  Los Angeles County       06037
3     PROVIDER_CA     S946274        2023-05-09            5.47                18961.0              RES    CA    49343  Los Angeles County       06037
4     PROVIDER_CA     S730237        2016-10-18            8.86                36094.0              RES    CA    84945  Los Angeles County       06037


### 1.2 Load ACS county data

ACS comes with a `GEO_ID` like `0500000US06037`. The last 5 digits are the
county FIPS, which is what we need for joining.


In [4]:
acs = pd.read_csv(RAW_DIR / "acs_county_housing_income.csv")
print(f"ACS county data: {acs.shape[0]:,} rows, {acs.shape[1]} columns")
print(acs.head().to_string())


ACS county data: 50 rows, 7 columns
           GEO_ID                              NAME  B19013_001E  B25001_001E  B25003_002E  B25035_001E  B01003_001E
0  0500000US06037    Los Angeles County, California        69384       703671       394973         1982      1787233
1  0500000US06059         Orange County, California        88041        77962        57603         1979       229538
2  0500000US06073      San Diego County, California        35066       591434       278937         1988      1302127
3  0500000US06075  San Francisco County, California        49917       669509       394802         1963      2141652
4  0500000US06081      San Mateo County, California        84418      1085326       498007         1978      2404333


### 1.3 Load EIA state-level electricity prices

EIA's API returns long-format state-year-sector rows. We filter to the
residential sector and join later on state code.


In [5]:
eia = pd.read_csv(RAW_DIR / "eia_state_electricity_prices.csv")
print(f"EIA state prices: {eia.shape[0]:,} rows, {eia.shape[1]} columns")
print(eia.head().to_string())


EIA state prices: 189 rows, 7 columns
   period stateid stateDescription sectorid   sectorName  price             price-units
0    2015      AZ          Arizona      RES  residential  12.63  cents per kilowatthour
1    2016      AZ          Arizona      RES  residential  12.12  cents per kilowatthour
2    2017      AZ          Arizona      RES  residential  13.19  cents per kilowatthour
3    2018      AZ          Arizona      RES  residential  11.84  cents per kilowatthour
4    2019      AZ          Arizona      RES  residential  13.60  cents per kilowatthour


## 2. Data Organization

Goal: confirm the project's file structure and review what we have on disk.

This notebook uses a flat layout for portability — the notebook and all
data files live together in one folder:

```
project_folder/
├── 01_data_wrangling.ipynb           <- this notebook
├── tracking_the_sun_sample.csv       <- raw input
├── acs_county_housing_income.csv     <- raw input
├── eia_state_electricity_prices.csv  <- raw input
└── county_solar_adoption.csv         <- generated by this notebook
```

A more conventional project would split inputs and outputs into
`data/raw/` and `data/processed/` subfolders. That's better practice
for larger projects but creates path-resolution friction when the
notebook is shared on its own. The flat layout keeps the notebook
self-contained, and reorganizing into subfolders later requires only
changing two `Path()` lines in cell 1.


In [6]:
# Sanity check: list files using glob (per the assignment hint)
print("Files in working folder:")
for f in sorted(glob.glob(str(RAW_DIR / "*.csv"))):
    print(" ", Path(f).name)


Files in working folder:
  acs_county_housing_income.csv
  eia_state_electricity_prices.csv
  tracking_the_sun_sample.csv


## 3. Data Definition

Goal: understand each dataset's columns, types, and ranges before cleaning.


### 3.1 Tracking the Sun

In [7]:
print("Columns and dtypes:")
print(tts.dtypes)
print(f"\nShape: {tts.shape}")


Columns and dtypes:
data_provider_1              str
system_id_1                  str
installation_date            str
system_size_DC           float64
total_installed_price    float64
customer_segment             str
state                        str
zip_code                  string
county                       str
county_fips               string
dtype: object

Shape: (12724, 10)


In [8]:
print("Tracking the Sun — summary statistics for numeric columns:")
print(tts.describe(include=[np.number]).to_string())


Tracking the Sun — summary statistics for numeric columns:
       system_size_DC  total_installed_price
count    12724.000000           1.122600e+04
mean         7.772673           3.929615e+04
std          7.577342           1.833338e+05
min         -1.240000          -5.358000e+03
25%          5.780000           2.121000e+04
50%          7.485000           2.822300e+04
75%          9.190000           3.562550e+04
max        246.590000           4.960900e+06


In [9]:
print("Tracking the Sun — unique value counts for categorical columns:")
for col in ["data_provider_1", "customer_segment", "state"]:
    print(f"\n  {col} ({tts[col].nunique()} unique values):")
    print(tts[col].value_counts(dropna=False).head(10).to_string())


Tracking the Sun — unique value counts for categorical columns:

  data_provider_1 (21 unique values):
data_provider_1
PROVIDER_CA    2437
PROVIDER_NY    1232
PROVIDER_TX    1091
PROVIDER_AZ     993
PROVIDER_FL     954
PROVIDER_NJ     734
PROVIDER_GA     526
PROVIDER_OR     503
PROVIDER_MD     497
PROVIDER_IL     435

  customer_segment (2 unique values):
customer_segment
RES    12111
COM      613

  state (21 unique values):
state
CA    2437
NY    1232
TX    1091
AZ     993
FL     954
NJ     734
GA     526
OR     503
MD     497
IL     435


### 3.2 ACS county data

The ACS column codes are not human-readable. Rename now so ACS code is
self-documenting.

| ACS code | Meaning |
|---|---|
| B19013_001E | Median household income |
| B25001_001E | Total housing units |
| B25003_002E | Owner-occupied housing units |
| B25035_001E | Median year structure built |
| B01003_001E | Total population |


In [10]:
acs = acs.rename(columns={
    "B19013_001E": "median_household_income",
    "B25001_001E": "total_housing_units",
    "B25003_002E": "owner_occupied_units",
    "B25035_001E": "median_year_built",
    "B01003_001E": "population",
})
print("ACS columns after rename:")
print(acs.columns.tolist())
print(f"\nACS shape: {acs.shape}")
print("\nACS dtypes:")
print(acs.dtypes)


ACS columns after rename:
['GEO_ID', 'NAME', 'median_household_income', 'total_housing_units', 'owner_occupied_units', 'median_year_built', 'population']

ACS shape: (50, 7)

ACS dtypes:
GEO_ID                       str
NAME                         str
median_household_income    int64
total_housing_units        int64
owner_occupied_units       int64
median_year_built          int64
population                 int64
dtype: object


In [11]:
print("ACS — summary statistics:")
print(acs.describe().to_string())


ACS — summary statistics:
       median_household_income  total_housing_units  owner_occupied_units  median_year_built    population
count             5.000000e+01         5.000000e+01             50.000000       5.000000e+01  5.000000e+01
mean             -2.659475e+07         6.173512e+05         383760.540000      -1.333140e+07  1.609815e+06
std               1.319806e+08         3.119927e+05         207713.740199       9.428118e+07  8.441041e+05
min              -6.666667e+08         7.796200e+04          57603.000000      -6.666667e+08  2.295380e+05
25%               5.405300e+04         3.673668e+05         203929.250000       1.964250e+03  9.131265e+05
50%               7.413800e+04         6.144145e+05         390054.500000       1.973000e+03  1.669752e+06
75%               9.200875e+04         8.527638e+05         521131.000000       1.981750e+03  2.253852e+06
max               1.271410e+05         1.191843e+06         792894.000000       2.010000e+03  3.605871e+06


### 3.3 EIA state prices

In [12]:
print("EIA columns and dtypes:")
print(eia.dtypes)
print(f"\nEIA shape: {eia.shape}")
print(f"\nUnique states: {eia['stateid'].nunique()}")
print(f"Year range: {eia['period'].min()} to {eia['period'].max()}")
print(f"Sectors present: {eia['sectorid'].unique().tolist()}")


EIA columns and dtypes:
period                int64
stateid                 str
stateDescription        str
sectorid                str
sectorName              str
price               float64
price-units             str
dtype: object

EIA shape: (189, 7)

Unique states: 21
Year range: 2015 to 2023
Sectors present: ['RES']


In [13]:
print("EIA price summary (cents/kWh):")
print(eia['price'].describe().to_string())


EIA price summary (cents/kWh):
count    189.000000
mean      16.253069
std        5.668393
min        9.030000
25%       12.520000
50%       14.480000
75%       17.360000
max       30.790000


## 4. Data Cleaning

This is where the bulk of the wrangling happens. Each subsection below
addresses a specific data issue surfaced during definition.


### 4.1 Tracking the Sun: filter to residential and drop bad rows

The proposal scopes the analysis to **residential** PV (solar panels). The raw data includes
some commercial rows that need to be dropped before aggregation.


In [14]:
print("Before residential filter:", len(tts))
print(tts['customer_segment'].value_counts(dropna=False).to_string())

tts_res = tts[tts['customer_segment'] == 'RES'].copy()
print("\nAfter residential filter:", len(tts_res))


Before residential filter: 12724
customer_segment
RES    12111
COM      613

After residential filter: 12111


### 4.2 Tracking the Sun: handle missing FIPS codes

A small number of installations have a missing `county_fips`. These rows can't
be aggregated to county level so they have to be dropped, but the count should
be small. If it isn't, that's a data-quality flag worth investigating.


In [15]:
missing_fips = tts_res['county_fips'].isna().sum()
pct_missing = 100 * missing_fips / len(tts_res)
print(f"Rows with missing county_fips: {missing_fips:,}  ({pct_missing:.2f}%)")

# Decision: drop rows missing FIPS. <0.5% of records, no way to recover.
tts_res = tts_res.dropna(subset=['county_fips']).copy()
print(f"After dropping: {len(tts_res):,} rows")


Rows with missing county_fips: 19  (0.16%)
After dropping: 12,092 rows


### 4.3 Tracking the Sun: handle exact duplicate rows

Real TTS releases occasionally include duplicate installation records from
overlapping data-provider feeds. Drop exact duplicates.


In [16]:
dupes = tts_res.duplicated().sum()
print(f"Exact duplicate rows: {dupes:,}")

tts_res = tts_res.drop_duplicates().copy()
print(f"After dedup: {len(tts_res):,} rows")


Exact duplicate rows: 49
After dedup: 12,043 rows


### 4.4 Tracking the Sun: handle outliers in cost and system size

Three outlier patterns to address:

1. **Zero cost** — clearly a data entry error; treat as missing
2. **Absurdly high cost** (>$200,000 for a residential system) — likely a unit error (cents vs dollars); treat as missing
3. **Oversized systems** (>30 kW) — almost certainly commercial misclassified as residential; drop

For cost, we *don't* drop — many rows have legitimate missing cost data and
we'll either impute later or use cost as an optional feature.


In [17]:
print("System size distribution before cleaning:")
print(tts_res['system_size_DC'].describe().to_string())

# Drop nonsensical sizes: must be > 0 and <= 30 kW (residential ceiling).
# Negative or zero values are data entry errors; >30 kW is almost certainly
# commercial misclassified as residential.
bad_size = ((tts_res['system_size_DC'] <= 0) | (tts_res['system_size_DC'] > 30)).sum()
neg_size = (tts_res['system_size_DC'] <= 0).sum()
oversized = (tts_res['system_size_DC'] > 30).sum()
print(f"\nNonsensical system sizes being dropped: {bad_size}")
print(f"  - zero or negative size: {neg_size}")
print(f"  - oversized (>30 kW):    {oversized}")
tts_res = tts_res[(tts_res['system_size_DC'] > 0) & (tts_res['system_size_DC'] <= 30)].copy()

# For cost: convert zero and absurd values to NaN (don't drop the row)
zero_cost = (tts_res['total_installed_price'] == 0).sum()
absurd_cost = (tts_res['total_installed_price'] > 200000).sum()
print(f"Zero-cost rows being set to NaN: {zero_cost}")
print(f"Absurd-cost (>$200k) rows being set to NaN: {absurd_cost}")

tts_res.loc[tts_res['total_installed_price'] == 0, 'total_installed_price'] = np.nan
tts_res.loc[tts_res['total_installed_price'] > 200000, 'total_installed_price'] = np.nan

# Cost-per-watt sanity column for downstream use
tts_res['cost_per_watt'] = tts_res['total_installed_price'] / (tts_res['system_size_DC'] * 1000)

print("\nSystem size distribution after cleaning:")
print(tts_res['system_size_DC'].describe().to_string())
print("\nCost-per-watt distribution (NaN where cost was missing/bad):")
print(tts_res['cost_per_watt'].describe().to_string())


System size distribution before cleaning:
count    12043.000000
mean         7.757647
std          7.396860
min         -1.240000
25%          5.780000
50%          7.480000
75%          9.190000
max        246.590000

Nonsensical system sizes being dropped: 40
  - zero or negative size: 20
  - oversized (>30 kW):    20
Zero-cost rows being set to NaN: 61
Absurd-cost (>$200k) rows being set to NaN: 37

System size distribution after cleaning:
count    12003.000000
mean         7.501314
std          2.498201
min          0.170000
25%          5.790000
50%          7.480000
75%          9.180000
max         18.320000

Cost-per-watt distribution (NaN where cost was missing/bad):
count    10494.000000
mean         3.855844
std          0.573939
min          1.857143
25%          3.444149
50%          3.851650
75%          4.267805
max          5.793389


### 4.5 Tracking the Sun: aggregate to county level

This is the key transformation: from one row per installation to one row per
county, which is the unit of analysis for the project.

Target: `installs_total` (cumulative installations) and `installed_kw_total`
(cumulative DC capacity). We'll convert these to per-capita rates after
joining ACS in step 4.7.


In [18]:
tts_county = tts_res.groupby('county_fips').agg(
    installs_total=('system_id_1', 'count'),
    installed_kw_total=('system_size_DC', 'sum'),
    median_system_size_kw=('system_size_DC', 'median'),
    median_cost_per_watt=('cost_per_watt', 'median'),
    state=('state', 'first'),
    county_name=('county', 'first'),
).reset_index()

print(f"County-level TTS dataset: {tts_county.shape}")
print(tts_county.head(8).to_string())


County-level TTS dataset: (50, 7)
  county_fips  installs_total  installed_kw_total  median_system_size_kw  median_cost_per_watt state           county_name
0       04013             475             3576.35                  7.460              3.921808    AZ       Maricopa County
1       04019             464             3446.79                  7.455              3.840216    AZ           Pima County
2       06037             494             3629.23                  7.360              3.794104    CA    Los Angeles County
3       06059             396             3075.80                  7.810              3.793589    CA         Orange County
4       06073             237             1814.67                  7.480              3.831360    CA      San Diego County
5       06075             228             1656.37                  6.940              3.870438    CA  San Francisco County
6       06081             519             3992.81                  7.720              3.853632    CA     

### 4.6 ACS: handle suppressed values (-666666666)

The U.S. Census uses negative sentinel codes (`-666666666`, `-999999999`, etc.)
to indicate suppressed estimates. These need to become real NaNs before any
arithmetic. Reference: https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html


In [19]:
# Replace any large negative sentinel values with NaN across numeric ACS columns
acs_numeric_cols = ['median_household_income', 'total_housing_units',
                    'owner_occupied_units', 'median_year_built', 'population']

before_na = acs[acs_numeric_cols].isna().sum().sum()
for col in acs_numeric_cols:
    acs.loc[acs[col] < 0, col] = np.nan
after_na = acs[acs_numeric_cols].isna().sum().sum()

print(f"Sentinel values converted to NaN: {after_na - before_na}")
print("\nNaN counts per ACS column:")
print(acs[acs_numeric_cols].isna().sum().to_string())


Sentinel values converted to NaN: 3

NaN counts per ACS column:
median_household_income    2
total_housing_units        0
owner_occupied_units       0
median_year_built          1
population                 0


### 4.7 ACS: extract county FIPS from GEO_ID

The `GEO_ID` field looks like `0500000US06037`. We need just the trailing
5-digit FIPS for the join key.


In [20]:
acs['county_fips'] = acs['GEO_ID'].str[-5:]
print("Sample of extracted FIPS codes:")
print(acs[['GEO_ID', 'county_fips', 'NAME']].head().to_string())

# Confirm none lost their leading zero
fips_lengths = acs['county_fips'].str.len().value_counts()
print(f"\nFIPS length distribution (should all be 5):\n{fips_lengths.to_string()}")


Sample of extracted FIPS codes:
           GEO_ID county_fips                              NAME
0  0500000US06037       06037    Los Angeles County, California
1  0500000US06059       06059         Orange County, California
2  0500000US06073       06073      San Diego County, California
3  0500000US06075       06075  San Francisco County, California
4  0500000US06081       06081      San Mateo County, California

FIPS length distribution (should all be 5):
county_fips
5    50


### 4.8 Merge TTS county aggregates with ACS

This is the central join. We use a **left join from ACS to TTS** because
ACS gives us the universe of counties — including ones with zero
installations, which are valid observations for adoption modeling (a
county with very low solar adoption is still a data point).


In [21]:
county = acs.merge(
    tts_county,
    on='county_fips',
    how='left',
    suffixes=('_acs', '_tts')
)

# Counties with no installations should have 0, not NaN, for installs_total / installed_kw_total
fill_zero = ['installs_total', 'installed_kw_total']
for col in fill_zero:
    county[col] = county[col].fillna(0)

print(f"Merged county dataset: {county.shape}")
print("\nJoin diagnostic:")
print(f"  Counties in ACS:          {len(acs)}")
print(f"  Counties in TTS aggregate: {len(tts_county)}")
print(f"  Counties in merged:        {len(county)}")
print(f"  Counties with 0 installs:  {(county['installs_total'] == 0).sum()}")


Merged county dataset: (50, 14)

Join diagnostic:
  Counties in ACS:          50
  Counties in TTS aggregate: 50
  Counties in merged:        50
  Counties with 0 installs:  0


### 4.9 Build the per-capita target variable

The proposal specifies the modeling target as **installations per 1,000
owner-occupied households**. We need owner-occupied count from ACS to
build it.


In [22]:
# Avoid divide-by-zero / divide-by-NaN
county['installs_per_1k_oo_hh'] = (
    county['installs_total'] / county['owner_occupied_units'] * 1000
)
county['kw_per_1k_oo_hh'] = (
    county['installed_kw_total'] / county['owner_occupied_units'] * 1000
)

print("Target variable distribution (installs per 1,000 owner-occupied HH):")
print(county['installs_per_1k_oo_hh'].describe().to_string())


Target variable distribution (installs per 1,000 owner-occupied HH):
count    50.000000
mean      1.150351
std       1.458633
min       0.113169
25%       0.334290
50%       0.565616
75%       1.198577
max       6.874642


### 4.10 Merge state-level electricity prices

EIA prices are state-year. For the cross-sectional model we'll use the
**most recent year** (2023). Time-varying versions can be built later.


In [23]:
# Filter to residential sector and most recent year
eia_recent = eia[(eia['sectorid'] == 'RES') & (eia['period'] == eia['period'].max())].copy()
eia_recent = eia_recent[['stateid', 'price']].rename(
    columns={'stateid': 'state', 'price': 'residential_price_cents_kwh'}
)
print(f"EIA most-recent-year slice: {eia_recent.shape}")
print(eia_recent.head().to_string())


EIA most-recent-year slice: (21, 2)
   state  residential_price_cents_kwh
8     AZ                        15.09
17    CA                        30.79
26    CO                        16.18
35    CT                        30.25
44    FL                        16.04


In [24]:
county = county.merge(eia_recent, on='state', how='left')

# Diagnose any unmatched states
unmatched = county[county['residential_price_cents_kwh'].isna()]
print(f"Counties unmatched on state-price join: {len(unmatched)}")
print(f"\nFinal merged shape: {county.shape}")


Counties unmatched on state-price join: 0

Final merged shape: (50, 17)


### 4.11 Final tidy and save

Drop intermediate columns, reorder for readability, and save to the
processed directory.


In [25]:
# Drop the raw GEO_ID and ACS code columns we no longer need
drop_cols = ['GEO_ID', 'NAME']
county = county.drop(columns=[c for c in drop_cols if c in county.columns])

# Reorder columns: identifiers, target, then features
preferred_order = [
    'county_fips', 'county_name', 'state',
    # targets
    'installs_total', 'installed_kw_total',
    'installs_per_1k_oo_hh', 'kw_per_1k_oo_hh',
    # features
    'median_household_income', 'owner_occupied_units', 'total_housing_units',
    'median_year_built', 'population',
    'residential_price_cents_kwh',
    'median_system_size_kw', 'median_cost_per_watt',
]
county = county[[c for c in preferred_order if c in county.columns]]

print(f"Final shape: {county.shape}")
print("\nFinal columns:")
print(county.dtypes.to_string())

print("\nFirst 10 rows of cleaned county dataset:")
print(county.head(10).to_string())


Final shape: (50, 15)

Final columns:
county_fips                     object
county_name                        str
state                              str
installs_total                   int64
installed_kw_total             float64
installs_per_1k_oo_hh          float64
kw_per_1k_oo_hh                float64
median_household_income        float64
owner_occupied_units           float64
total_housing_units            float64
median_year_built              float64
population                     float64
residential_price_cents_kwh    float64
median_system_size_kw          float64
median_cost_per_watt           float64

First 10 rows of cleaned county dataset:
  county_fips           county_name state  installs_total  installed_kw_total  installs_per_1k_oo_hh  kw_per_1k_oo_hh  median_household_income  owner_occupied_units  total_housing_units  median_year_built  population  residential_price_cents_kwh  median_system_size_kw  median_cost_per_watt
0       06037    Los Angeles County    CA   

In [26]:
print("Final summary statistics:")
print(county.describe().to_string())

print("\nMissing-value counts:")
print(county.isna().sum().to_string())


Final summary statistics:
       installs_total  installed_kw_total  installs_per_1k_oo_hh  kw_per_1k_oo_hh  median_household_income  owner_occupied_units  total_housing_units  median_year_built    population  residential_price_cents_kwh  median_system_size_kw  median_cost_per_watt
count       50.000000           50.000000              50.000000        50.000000                48.000000             50.000000         5.000000e+01          49.000000  5.000000e+01                    50.000000              50.000000             50.000000
mean       240.060000         1800.765400               1.150351         8.647319             74912.645833         383760.540000         6.173512e+05        1972.061224  1.609815e+06                    19.253800               7.488500              3.854701
std        121.942678          918.499179               1.458633        11.052005             23243.801090         207713.740199         3.119927e+05          15.430500  8.441041e+05                     

In [27]:
out_path = PROCESSED_DIR / "county_solar_adoption.csv"
county.to_csv(out_path, index=False)
print(f"Saved cleaned dataset to: {out_path}")
print(f"  Rows:    {len(county):,}")
print(f"  Columns: {len(county.columns)}")


Saved cleaned dataset to: county_solar_adoption.csv
  Rows:    50
  Columns: 15


## 5. Wrangling notes (for mentor review)

These map directly to the questions Springboard asks you to consider before
the next mentor call.

**What cleaning steps did I perform?**

1. Filtered Tracking the Sun to residential installations only (the project scope).
2. Forced `county_fips` to string on read so leading zeros (CA, CO, CT, etc.) were preserved.
3. Dropped rows with missing county FIPS — small share, unrecoverable.
4. Dropped exact duplicate installation rows from overlapping data-provider feeds.
5. Dropped systems with non-positive size (data entry errors) and systems above 30 kW (commercial misclassified as residential).
6. Converted zero-dollar and absurdly high installation costs to NaN (data entry / unit errors).
7. Aggregated installation-level data up to the county level — the unit of analysis.
8. Replaced ACS sentinel values (`-666666666`) with NaN before any arithmetic.
9. Extracted 5-digit county FIPS from the ACS `GEO_ID` field.
10. Left-joined ACS to TTS aggregates so counties with zero installations are kept (zeros, not nulls).
11. Built the `installs_per_1k_oo_hh` target using owner-occupied households as the denominator.
12. Joined state-level residential electricity prices from EIA, using the most recent year for a cross-sectional model.

**How did I deal with missing values?**

I treated different missingness patterns differently rather than applying one
blanket rule. Missing FIPS → drop (no way to recover, very rare). Missing cost
→ keep the row, set cost to NaN (cost is a secondary feature; many rows have
legitimately missing cost data and dropping them would bias the sample). ACS
suppressed estimates → NaN (will be handled at the modeling step, likely with
median imputation or a missing-indicator flag).

**Were there outliers, and how did I handle them?**

Two types. System size outliers: anything with a non-positive size (data entry
errors) and anything above 30 kW (almost certainly commercial misclassified as
residential) were dropped because those rows aren't recoverable as residential
observations. Cost outliers (zero or above $200k) were converted to NaN rather
than dropped, because the rest of the row's information is still valid and the
cost field is a feature, not the target.

**Open questions for the mentor call:**

- Whether to use installs *per 1,000 owner-occupied households* or per total
  housing units. The proposal commits to the former; worth confirming.
- Cross-sectional vs panel framing. This notebook builds a single most-recent-year
  snapshot. A panel using all years of EIA prices would let us study within-county
  change but adds modeling complexity.
- Imputation strategy for missing income / cost values — defer to the EDA step.

**Next step:** Exploratory Data Analysis on the cleaned county dataset, then
feature engineering (NSRDB irradiance, DSIRE policy indicators).
